<a href="https://colab.research.google.com/github/Radhakuchekar/Preparation/blob/pyspark/snapchat_time_spent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
from pyspark.sql.functions import *
spark

In [16]:

data = [
    (7274, 123, "open", 4.50, "06/22/2022 12:00:00"),
    (2425, 123, "send", 3.50, "06/22/2022 12:00:00"),
    (1413, 456, "send", 5.67, "06/23/2022 12:00:00"),
    (1413, 456, "send", 5.67, "06/23/2022 12:00:00"),
    (1414, 789, "chat", 11.00, "06/25/2022 12:00:00"),
    (2536, 456, "open", 3.00, "06/25/2022 12:00:00")
]
columns = ["activity_id", "user_id", "activity_type", "time_spent", "activity_date"]
df = spark.createDataFrame(data, schema=columns)
df.show()



+-----------+-------+-------------+----------+-------------------+
|activity_id|user_id|activity_type|time_spent|      activity_date|
+-----------+-------+-------------+----------+-------------------+
|       7274|    123|         open|       4.5|06/22/2022 12:00:00|
|       2425|    123|         send|       3.5|06/22/2022 12:00:00|
|       1413|    456|         send|      5.67|06/23/2022 12:00:00|
|       1413|    456|         send|      5.67|06/23/2022 12:00:00|
|       1414|    789|         chat|      11.0|06/25/2022 12:00:00|
|       2536|    456|         open|       3.0|06/25/2022 12:00:00|
+-----------+-------+-------------+----------+-------------------+



In [17]:
# Data for age_breakdown
age_data = [
    (123, "31-35"),
    (456, "26-30"),
    (789, "21-25")
]
age_columns = ["user_id", "age_bucket"]
age_df = spark.createDataFrame(age_data, schema=age_columns)
age_df.show()

+-------+----------+
|user_id|age_bucket|
+-------+----------+
|    123|     31-35|
|    456|     26-30|
|    789|     21-25|
+-------+----------+



In [24]:
df_with_age = df.join(age_df, ['user_id'], 'inner')
df_with_age

user_id,activity_id,activity_type,time_spent,activity_date,age_bucket
123,7274,open,4.5,06/22/2022 12:00:00,31-35
123,2425,send,3.5,06/22/2022 12:00:00,31-35
456,1413,send,5.67,06/23/2022 12:00:00,26-30
456,1413,send,5.67,06/23/2022 12:00:00,26-30
456,2536,open,3.0,06/25/2022 12:00:00,26-30
789,1414,chat,11.0,06/25/2022 12:00:00,21-25


In [25]:
from pyspark.sql import Window
window_spec = Window.partitionBy(['age_bucket', 'activity_type'])
agg_by_age_df = df_with_age.withColumn("sum", sum(col("time_spent")).over(window_spec))
agg_by_age_df

user_id,activity_id,activity_type,time_spent,activity_date,age_bucket,sum
789,1414,chat,11.0,06/25/2022 12:00:00,21-25,11.0
456,2536,open,3.0,06/25/2022 12:00:00,26-30,3.0
456,1413,send,5.67,06/23/2022 12:00:00,26-30,11.34
456,1413,send,5.67,06/23/2022 12:00:00,26-30,11.34
123,7274,open,4.5,06/22/2022 12:00:00,31-35,4.5
123,2425,send,3.5,06/22/2022 12:00:00,31-35,3.5


In [26]:
agg_by_age_df = agg_by_age_df.groupBy(['activity_type', 'age_bucket']).agg(max(col("sum")).alias("time_spent"))

In [27]:
agg_by_age_df = agg_by_age_df.filter("activity_type in ('open','send')").groupBy("age_bucket").pivot("activity_type").agg(first("time_spent"))

agg_by_age_df

age_bucket,open,send
31-35,4.5,3.5
26-30,3.0,11.34


In [36]:
agg_by_age_df.withColumn("open_perc", round(col("open") / (col("open")+ col("send"))*100,2)).withColumn("send_perc", round(col("send") / (col("open")+ col("send"))*100, 2)).select(['age_bucket', 'open_perc', 'send_perc'])


age_bucket,open_perc,send_perc
31-35,56.25,43.75
26-30,20.92,79.08
